# Calgary Spatial ETL: Big-Picture Knowledge Assessment

This cumulative workbook has two purposes:

1. Inspect the current repository with safe, read-only checks.
2. Test whether you can explain and apply the complete workflow from environment setup through Git, Extract, Transform, QA/QC, Load, verification, and maintenance.

## How to use this notebook

- Use the `calgary-etl` kernel.
- Work from top to bottom.
- Write answers before opening the answer key.
- Code exercises use toy or temporary data and do not write to production PostGIS.
- Run self-check cells only after completing the preceding exercise.
- A failed assertion is feedback, not damage to the project.
- Do not score your own written or open-ended code work. Save the completed notebook and ask AI to grade it using the final rubric.

## Scoring: 100 points

| Area | Points |
|---|---:|
| Big picture and workflow order | 10 |
| Environment and dependencies | 10 |
| Git and secure collaboration | 10 |
| Extract | 10 |
| Transform | 15 |
| QA/QC | 15 |
| Load and PostGIS | 15 |
| Testing, orchestration, and troubleshooting | 15 |

Objective questions are scored automatically. AI grades the open-ended code and written responses, cites evidence, and calculates the final result.

Suggested interpretation after AI review: **90-100 ready to explain and operate**, **75-89 solid with targeted review**, **60-74 developing**, **below 60 revisit the relevant practice modules**.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import tempfile

import geopandas as gpd
from shapely.geometry import Point, Polygon


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in [candidate, *candidate.parents]:
        if (directory / "src").is_dir() and (directory / "environment.yml").exists():
            return directory
    raise FileNotFoundError("Run this notebook from inside calgary-spatial-etl.")


PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Version: {platform.python_version()}")

## 1. Inspect the Repository and Identify Project Components

Before judging completeness, identify the evidence. Classify the repository's production source, tests, infrastructure, environment definitions, generated data, documentation, and learning resources.

**Written response:** Why is the presence of a file not enough to prove that its behavior works?

> Your answer:

## 2. Validate Environment Setup and Dependencies

Record the active interpreter, Python version, Conda environment, and key GIS package versions. Explain why compiled GIS dependencies benefit from a declared environment. Never print secret environment-variable values.

**Written response:** Distinguish the interpreter, environment, package manifest, and environment variable.

> Your answer:

## 3. Inspect Git Configuration and Repository State

Use only read-only commands here. Interpret the branch, latest commit, remotes, saved modifications, and ignored generated files.

**Written response:** Explain the difference among an editor's unsaved buffer, Git's working tree, staging area, and last commit.

> Your answer:

## 4. Map the End-to-End Data Workflow

Write the safe operational order and state the input, output, and guarantee added by each stage. Environment, Git, database setup, and tests support ETL but are not all data-processing stages.

**Ordering response:**

> Your ordered stages:

In [ ]:
expected_files = [
    "environment.yml",
    ".gitignore",
    "docker-compose.yml",
    "sql/init.sql",
    "src/config.py",
    "src/extract.py",
    "src/transform.py",
    "src/qa.py",
    "src/load.py",
    "src/main.py",
    "tests/test_qa.py",
    "tests/test_load_integration.py",
]

file_status = {
    relative_path: (PROJECT_ROOT / relative_path).exists()
    for relative_path in expected_files
}
package_versions = {
    package: importlib.metadata.version(package)
    for package in ["geopandas", "shapely", "pyproj", "sqlalchemy", "geoalchemy2", "psycopg2"]
}

def read_only_git(*arguments: str) -> str:
    result = subprocess.run(
        ["git", *arguments],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()

print("Required files:")
for path, exists in file_status.items():
    print(f"  {'PASS' if exists else 'MISSING'} {path}")
print("\nPackage versions:", package_versions)
print("\nBranch:", read_only_git("branch", "--show-current"))
print("Latest commit:", read_only_git("log", "-1", "--oneline"))
print("Saved changes:\n", read_only_git("status", "--short") or "(clean)")
print("Remotes:\n", read_only_git("remote", "-v") or "(none configured)")

In [ ]:
# Exercise 1: Put the lifecycle in safe order.
# Replace the list with the correct sequence.
workflow_order = [
    "TODO",
]

expected_workflow_stages = {
    "environment setup",
    "git preparation",
    "postgis setup",
    "extract",
    "transform",
    "qa/qc",
    "load",
    "post-load verification",
    "automated tests",
    "review and commit",
}

assert set(workflow_order) == expected_workflow_stages, "Use every stage exactly once."
assert workflow_order.index("extract") < workflow_order.index("transform")
assert workflow_order.index("transform") < workflow_order.index("qa/qc")
assert workflow_order.index("qa/qc") < workflow_order.index("load")
assert workflow_order.index("load") < workflow_order.index("post-load verification")
print("PASS: the required stage dependencies are in a safe order.")

## 5. Validate Data Extraction

For this project, extraction means HTTP GeoJSON download, status validation, timeout protection, stable raw paths, raw preservation, and provenance logging. Think beyond the happy path: schema drift, service failure, pagination, authentication, retries, and rate limits are general extraction concerns even when a particular endpoint does not require all of them.

**Scenario:** One Calgary endpoint returns HTTP 503 after two files were downloaded. What should the stage report, what artifacts can be trusted, and why should Transform not begin as though the snapshot were complete?

> Your answer:

## 6. Validate Data Transformation

Transformation adds a stable schema and spatial contract. Explain column normalization, field selection, ID typing, geometry cleaning, active geometry preservation, assigning versus transforming a CRS, deterministic behavior, and separation of raw and processed files.

**Scenario:** A GeoJSON has longitude/latitude coordinates but no CRS metadata. Explain the two distinct operations needed to produce valid `EPSG:3347` coordinates and the danger of confusing them.

> Your answer:

In [ ]:
# Exercise 2: Model an extraction provenance record without making a network request.
# Fill every TODO with an appropriate value.
toy_payload = b'{"type":"FeatureCollection","features":[]}'
extraction_record = {
    "dataset": "TODO",
    "source_url": "TODO",
    "output_path": "TODO",
    "http_status": None,  # TODO
    "bytes_written": None,  # TODO
    "downloaded_at_utc": "TODO",
}

assert extraction_record["dataset"] != "TODO"
assert extraction_record["source_url"].startswith("https://")
assert extraction_record["output_path"].startswith("data/raw/")
assert extraction_record["http_status"] == 200
assert extraction_record["bytes_written"] == len(toy_payload)
assert extraction_record["downloaded_at_utc"].endswith("+00:00") or extraction_record["downloaded_at_utc"].endswith("Z")
print("PASS: the record captures source, destination, status, size, and UTC retrieval time.")

In [ ]:
# Exercise 3: Complete a toy spatial transformation.
raw_gdf = gpd.GeoDataFrame(
    {
        "Feature ID": [101, 102, 103],
        "Site Name": ["North", "Centre", "South"],
    },
    geometry=[Point(-114.1, 51.1), Point(-114.0, 51.05), Point(-113.95, 51.0)],
    crs="EPSG:4326",
)

# TODO: Make a copy.
# TODO: Rename attributes to feature_id and site_name while retaining geometry.
# TODO: Cast feature_id to pandas string dtype.
# TODO: Reproject to EPSG:3347 and assign the result to transformed_gdf.
transformed_gdf = None

assert isinstance(transformed_gdf, gpd.GeoDataFrame)
assert list(transformed_gdf.columns) == ["feature_id", "site_name", "geometry"]
assert str(transformed_gdf["feature_id"].dtype) == "string"
assert transformed_gdf.crs.to_string() == "EPSG:3347"
assert len(transformed_gdf) == len(raw_gdf)
assert raw_gdf.crs.to_string() == "EPSG:4326", "Do not mutate the raw input."
print("PASS: schema, ID type, geometry, row count, CRS, and raw-data separation are correct.")

## 7. Run Data QA/QC Checks

QA/QC must independently test processed outputs for completeness, validity, uniqueness, consistency, and fitness for publication. Classify checks as blocking failures, warnings, or diagnostics. In this project, missing files or fields, zero rows, geometry defects, ID defects, and the wrong CRS block loading.

**Written response:** Why should QA report a geometry defect instead of repairing it? Distinguish QA from the Transform responsibility.

> Your answer:

**Generalization:** For a different ETL project, describe checks for data types, ranges, referential integrity, freshness, null rates, row reconciliation, and distribution change. Which would block publication, and why?

> Your answer:

## 8. Validate Data Loading

A trustworthy load aligns target schema and geometry metadata, uses deliberate append/replace/upsert semantics, preserves transaction atomicity, supports safe reruns, and reconciles the committed destination. This project replaces tables in one transaction, then checks row counts, SRID, and GIST indexes.

**Scenario:** Layer three of four raises an exception. Describe the expected database state when all four loads share one transaction, and contrast it with four unrelated commits.

> Your answer:

**Design question:** When would append or upsert be preferable to replace? State the key, deduplication, batch, and recovery requirements that would become necessary.

> Your answer:

In [ ]:
# Exercise 4: Classify controlled layers with the production QA inspector.
from src.qa import inspect_layer

with tempfile.TemporaryDirectory() as temporary_directory:
    temporary_path = Path(temporary_directory)
    valid_path = temporary_path / "valid.geojson"
    duplicate_path = temporary_path / "duplicate.geojson"

    valid_layer = gpd.GeoDataFrame(
        {"feature_id": ["A", "B"], "name": ["Alpha", "Beta"]},
        geometry=[Point(0, 0), Point(1, 1)],
        crs="EPSG:3347",
    )
    duplicate_layer = valid_layer.copy()
    duplicate_layer["feature_id"] = ["A", "A"]
    valid_layer.to_file(valid_path, driver="GeoJSON")
    duplicate_layer.to_file(duplicate_path, driver="GeoJSON")

    valid_result = inspect_layer("valid", valid_path, ["feature_id", "name"], "feature_id")
    duplicate_result = inspect_layer("duplicate", duplicate_path, ["feature_id", "name"], "feature_id")

# Predict these values before running the assertions.
assert valid_result.passed is True
assert duplicate_result.passed is False
assert duplicate_result.duplicate_id_count == 1
assert valid_result.crs == "EPSG:3347"
print("PASS: controlled QA accepts the valid layer and blocks the duplicate ID.")

## 9. Run Automated Tests and Static Analysis

Tests are executable evidence, not a guarantee about every possible input. Unit tests isolate QA behavior. Opt-in integration tests exercise GeoPandas, SQLAlchemy, and live PostGIS replacement and rollback. Static analysis, formatting, secret scanning, and notebook validation are separate checks and should be reported as required, optional, unavailable, passed, or failed.

**Written response:** Why are both happy-path and failure-path tests required for a data pipeline?

> Your answer:

## 10. Execute an End-to-End Smoke Test

A smoke test should use the smallest representative path that proves stage integration while controlling side effects. A real operational verification can run `python -m src.main`; a safe learner test can use fixtures, temporary files, and an isolated database schema.

**Plan response:** Design a smoke test. State the sample, stage order, row-count evidence, logs, timing, cleanup, and failure behavior.

> Your answer:

## 11. Assess Project Completeness and Expected Behavior

Use separate conclusions:

- **Complete:** required implementation and documentation exist.
- **Operational:** current runtime checks pass in the available environment.
- **Partially verified:** some checks were skipped or external dependencies were unavailable.
- **Blocked:** a required prerequisite or behavior failed.

Never infer operational status only from file presence.

## 12. Generate or Review the Project Markdown Summary

The companion guide is `learning/guides/project1_big_picture_guide.md`. It covers purpose, architecture, prerequisites, order, artifacts, QA controls, testing, completion criteria, limitations, and operations without explaining code line by line.

**Review response:** Identify one fact in the guide that should be updated whenever implementation behavior changes.

> Your answer:

In [ ]:
# Read-only readiness evidence. This runs deterministic tests only;
# live PostGIS integration tests remain an explicit operator choice.
test_result = subprocess.run(
    [sys.executable, "-m", "unittest", "tests.test_qa", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

guide_path = PROJECT_ROOT / "learning/guides/project1_big_picture_guide.md"
readiness_evidence = {
    "required_files_present": all(file_status.values()),
    "key_imports_available": len(package_versions) == 6,
    "deterministic_tests_pass": test_result.returncode == 0,
    "big_picture_guide_present": guide_path.exists(),
    "git_repository_present": (PROJECT_ROOT / ".git").is_dir(),
}

for check, passed in readiness_evidence.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")
print("\nTest output:\n", test_result.stderr.strip() or test_result.stdout.strip())

if all(readiness_evidence.values()):
    print("\nRepository readiness checks passed. Live integrations still require separate evidence.")

# Knowledge Test

Complete Sections 13-19 without opening Section 20. Enter one lowercase option letter for each objective question in the next code cell. Written prompts are graded with the rubric and answer key at the bottom.

## 13. Test Environment and Dependency Knowledge

1. Which artifact is the primary reproducible Conda definition?  
   a. `outputs/qa/qa_report.csv`  
   b. `environment.yml`  
   c. `.git/config`  
   d. `data/raw/`

2. `which python` points outside the intended environment. What should happen first?  
   a. edit Transform  
   b. delete GeoJSON  
   c. activate/select `calgary-etl`  
   d. commit the current state

3. Where should a nonpublic database password be supplied?  
   a. committed in `src/load.py`  
   b. in an ignored environment variable or secret store  
   c. in a notebook output  
   d. in the commit message

**Written:** Explain why a successful import and a declared dependency are different kinds of evidence.

> Your answer:

## 14. Test Git Workflow Knowledge

4. Which command reviews exactly what the next commit would contain?  
   a. `git diff --cached`  
   b. `git remote -v`  
   c. `git clean -fd`  
   d. `git log -1`

5. A generated file was committed before it was added to `.gitignore`. What is true?  
   a. `.gitignore` automatically deletes its history  
   b. it remains tracked until explicitly removed from tracking  
   c. Git encrypts it  
   d. the branch becomes invalid

6. What is the safest order for a normal source change?  
   a. push, edit, test, stage  
   b. edit, stage everything, skip review, force push  
   c. edit, test, inspect diff, stage intended files, inspect staged diff, commit, push  
   d. commit, then create the environment

**Written:** Explain branches, remotes, pull requests, and how you would approach a merge conflict without discarding another person's work.

> Your answer:

## 15. Test Extraction Knowledge

7. Why call `raise_for_status()` after an HTTP request?  
   a. reproject geometry  
   b. turn unsuccessful HTTP responses into visible failures  
   c. create a GIST index  
   d. stage a commit

8. Why preserve raw snapshots separately from processed files?  
   a. to hide errors  
   b. to support provenance, replay, and comparison without mutating source evidence  
   c. because PostGIS cannot store geometry  
   d. to avoid all schema drift

9. A large paginated API changes while pages are being downloaded. Name the principal risk.  
   a. inconsistent snapshot with missing or repeated records  
   b. invalid Git branch  
   c. wrong Python interpreter  
   d. missing GIST index

**Written:** Design extraction observability for timeout, retry, rate-limit, pagination/checkpoint, schema drift, and byte/row-count behavior.

> Your answer:

## 16. Test Transformation Knowledge

10. What does `set_crs` do when used correctly?  
    a. declares what existing coordinates mean  
    b. calculates coordinates in a new CRS  
    c. repairs polygons  
    d. writes PostGIS

11. What does `to_crs` do?  
    a. renames fields  
    b. transforms coordinates into another CRS  
    c. removes duplicates  
    d. assigns Git authorship

12. Why are numeric-looking identifiers often strings?  
    a. IDs are labels; arithmetic and loss of formatting are undesirable  
    b. strings are always smaller  
    c. databases reject integers  
    d. GeoJSON has no numbers

13. Which statement is correct?  
    a. null, empty, and invalid geometry mean exactly the same thing  
    b. geometry can be dropped during field selection  
    c. repair is attempted in Transform and acceptance is independently checked in QA  
    d. CRS is an ordinary attribute column

**Written:** Explain deterministic transformations, missing-value policy, deduplication, joins, aggregation, business rules, and how each should be tested when added to a future pipeline.

> Your answer:

## 17. Test QA/QC Knowledge

14. Why must QA execute before Load?  
    a. to prevent known-bad data from being published  
    b. to speed up Git  
    c. to create the Conda environment  
    d. to download sources

15. Which check tests uniqueness?  
    a. CRS equality  
    b. duplicate configured IDs  
    c. HTTP status  
    d. Python version

16. A distribution-change warning detects an unusual but possibly legitimate shift. What is generally appropriate?  
    a. silently delete rows  
    b. report the diagnostic and apply an agreed severity policy  
    c. alter raw data  
    d. force every warning to pass

**Written:** Design a QA report row containing check name, dataset, severity, measured value, threshold, pass/fail/warning status, and diagnostic detail. Explain completeness, validity, consistency, uniqueness, freshness, referential integrity, and reconciliation.

> Your answer:

## 18. Test Loading Knowledge

17. What does one transaction across all layer loads provide?  
    a. pagination  
    b. all commit together or all roll back  
    c. automatic API retries  
    d. Git history

18. Why is repeated replacement idempotent for this snapshot pipeline?  
    a. each run appends duplicates  
    b. the destination is refreshed to the current complete snapshot  
    c. tables are never written  
    d. QA is skipped

19. Why verify a GIST geometry index?  
    a. it accelerates spatial query planning and filtering  
    b. it stores passwords  
    c. it replaces the CRS  
    d. it validates HTTP

20. Which evidence best reconciles a loaded layer?  
    a. the command printed a line  
    b. source and target counts match, SRID is valid, index exists, and transaction committed  
    c. a notebook tab is open  
    d. `.gitignore` exists

**Written:** Compare insert, append, replace, and upsert. Include keys, constraints, batches, rollback, idempotency, and post-load validation.

> Your answer:

## 19. Test Workflow Ordering and Troubleshooting Skills

For each case, state the **next diagnostic step**, not a random repair:

1. `ModuleNotFoundError: geopandas` before Extract.
2. Extract receives HTTP 503.
3. Transform reports a large unexpected row decrease.
4. QA reports `EPSG:4326` instead of `EPSG:3347`.
5. Load cannot connect to `localhost:5433`.
6. Source and loaded row counts differ.
7. Unit tests pass but PostGIS tests were skipped.
8. `git status` lists a credential file.
9. A merge conflict appears in a notebook.

> Your diagnostic responses:

In [ ]:
# Enter one lowercase answer letter for each objective question.
objective_responses = {
    1: "",
    2: "",
    3: "",
    4: "",
    5: "",
    6: "",
    7: "",
    8: "",
    9: "",
    10: "",
    11: "",
    12: "",
    13: "",
    14: "",
    15: "",
    16: "",
    17: "",
    18: "",
    19: "",
    20: "",
}

valid_choices = {"a", "b", "c", "d"}
assert all(answer in valid_choices for answer in objective_responses.values()), "Answer all 20 questions with a, b, c, or d."
print("All objective responses are complete. Continue to Section 20 when your written and code work is finished.")

## Capstone: Explain the Complete System

Without consulting the guide, explain the project to a new GIS developer from start to finish. Your explanation must include:

- the business purpose and four datasets
- environment reproducibility and secrets
- Git's role before, during, and after work
- Docker/PostGIS initialization
- every ETL stage in order
- each stage's input, output, and guarantee
- raw versus processed data
- geometry and CRS handling
- the QA/QC blocking decision
- transaction, idempotency, SRID, GIST index, and reconciliation
- unit, integration, smoke, and operational verification
- failure ownership and the first diagnostic step at each boundary
- generated artifacts versus versioned source

> Your capstone response:

### Capstone rubric: 10 of the 30 written-response points

The AI grader awards 1 point for each complete topic group above. A strong answer explains dependencies and reasons, not just command names.

# 20. Calculate Objective Results and Prepare AI Review

Open this section only after completing Sections 1-19.

## Objective answer key and feedback

1. **b** - `environment.yml` declares the Conda environment.
2. **c** - correct interpreter selection precedes code diagnosis.
3. **b** - secrets belong in ignored environment configuration or a secret store.
4. **a** - `git diff --cached` displays staged content.
5. **b** - ignore rules do not untrack existing history.
6. **c** - test and review both unstaged and staged intent before committing.
7. **b** - unsuccessful HTTP responses must become failures.
8. **b** - raw preservation supports provenance and replay.
9. **a** - changing pagination can create inconsistent snapshots.
10. **a** - `set_crs` assigns meaning to existing coordinates.
11. **b** - `to_crs` calculates coordinates in another CRS.
12. **a** - identifiers are labels rather than quantities.
13. **c** - Transform repairs; QA independently accepts or rejects.
14. **a** - QA blocks known defects before publication.
15. **b** - duplicate-ID checks measure uniqueness.
16. **b** - diagnostics need an explicit severity policy.
17. **b** - one transaction provides all-or-nothing publication.
18. **b** - replacement recreates the current snapshot without accumulated duplicates.
19. **a** - GIST indexes support efficient spatial queries.
20. **b** - committed, reconciled database evidence is stronger than console text.

## AI written-response rubric: 30 points

- **20 points:** AI assigns up to 2 points for each major written domain: evidence/completeness, environment, Git, Extract, Transform, QA/QC, Load, tests/smoke testing, troubleshooting, and summary maintenance.
- **10 points:** AI applies the capstone rubric above.
- Full credit requires the purpose and consequence, not only a definition.
- Partial credit is appropriate when the concept is right but stage ownership or evidence is missing.

## AI code rubric: 30 points

- Workflow ordering: 6 points
- Extraction provenance record: 6 points
- Toy GeoDataFrame transformation: 10 points
- Controlled production QA exercise: 8 points

AI awards full credit when all assertions pass and the recorded explanation demonstrates understanding. It may deduct when assertions pass through guessing but the explanation is incomplete. The learner does not enter these rubric scores.

In [ ]:
# Run only after completing the assessment.
objective_key = {
    1: "b", 2: "c", 3: "b", 4: "a", 5: "b",
    6: "c", 7: "b", 8: "b", 9: "a", 10: "a",
    11: "b", 12: "a", 13: "c", 14: "a", 15: "b",
    16: "b", 17: "b", 18: "b", 19: "a", 20: "b",
}

objective_points = sum(
    2 for question, expected in objective_key.items()
    if objective_responses.get(question) == expected
)

print(f"Objective subtotal: {objective_points}/40")
print("Code: awaiting AI review (30 points)")
print("Written and capstone: awaiting AI review (30 points)")
print("Final score: awaiting AI review of the saved notebook.")

stage_question_groups = {
    "environment": [1, 2, 3],
    "git": [4, 5, 6],
    "extract": [7, 8, 9],
    "transform": [10, 11, 12, 13],
    "qa_qc": [14, 15, 16],
    "load": [17, 18, 19, 20],
}
for stage, questions in stage_question_groups.items():
    correct = sum(
        objective_responses.get(number) == objective_key[number]
        for number in questions
    )
    print(f"{stage}: {correct}/{len(questions)} objective questions")

## AI Grading and Prioritized Remediation

After completing and saving the notebook, ask Copilot:

> Grade my completed `learning/assessments/project1_big_picture_assessment.ipynb`. Preserve my original answers and code. Use the notebook's 30-point code rubric and 30-point written/capstone rubric, verify the automatic objective subtotal, and cite specific notebook evidence for each award or deduction. Report the final score out of 100, strengths, misconceptions, and a prioritized remediation plan. Ask targeted follow-up questions before supplying complete corrected answers.

Use the AI feedback to answer:

1. Which stage can you explain without notes?
2. Which stage's input/output boundary is least clear?
3. Which failed objective questions form a pattern?
4. Which code exercise could you not complete independently?
5. Can you explain why the stage order protects data quality?
6. What evidence would you gather before claiming the project is operational on another machine?

> Your reflection:

Create a study plan in this order:

1. Repair prerequisites first: environment, interpreter, Git safety, and PostGIS setup.
2. Repair stage-order and boundary misunderstandings.
3. Revisit the matching notebook in `learning/starters/`.
4. Redo failed code exercises from a fresh kernel.
5. Explain the failed concept aloud using input, guarantee, output, and evidence.
6. Retake only the weak stage after one day, then retake the complete assessment later.

> Your prioritized plan:

A useful final standard is not memorizing every function. It is being able to operate the project safely, predict what each stage guarantees, identify which stage owns a defect, and gather evidence before declaring success.